In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

%matplotlib inline

In [ ]:
# Load results
import sys
sys.path.insert(0, '..')
from experiments.analyze_results import load_results, results_to_dataframe

RESULTS_DIR = '../results'  # Change to your results directory

results = load_results(RESULTS_DIR)
df = results_to_dataframe(results)
print(f"Loaded {len(df)} experiment results")
df.head()

## 1. Overall Performance Summary

In [ ]:
# Summary statistics
print("Overall Statistics:")
print(f"  Mean Accuracy: {df['accuracy'].mean():.2%}")
print(f"  Std Accuracy:  {df['accuracy'].std():.2%}")
print(f"  Best:          {df['accuracy'].max():.2%}")
print(f"  Worst:         {df['accuracy'].min():.2%}")
print()
print("Best configuration:")
best = df.loc[df['accuracy'].idxmax()]
print(f"  {best['prompting']} + {best['decoding']} on {best['dataset']}")

## 2. Prompting Strategy Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy by prompting
prompting_stats = df.groupby('prompting')['accuracy'].agg(['mean', 'std']).sort_values('mean', ascending=False)
ax = axes[0]
bars = ax.bar(prompting_stats.index, prompting_stats['mean'], yerr=prompting_stats['std'], capsize=5)
ax.set_xlabel('Prompting Strategy')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy by Prompting Strategy')
ax.set_ylim(0, 1)

# Reasoning steps by prompting
ax = axes[1]
reasoning_stats = df.groupby('prompting')['avg_reasoning_steps'].mean().sort_values(ascending=False)
ax.bar(reasoning_stats.index, reasoning_stats.values, color='coral')
ax.set_xlabel('Prompting Strategy')
ax.set_ylabel('Avg Reasoning Steps')
ax.set_title('Reasoning Depth by Prompting Strategy')

plt.tight_layout()
plt.show()

## 3. Decoding Strategy Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

decoding_stats = df.groupby('decoding')['accuracy'].agg(['mean', 'std']).sort_values('mean', ascending=False)
bars = ax.bar(decoding_stats.index, decoding_stats['mean'], yerr=decoding_stats['std'], capsize=5, color='steelblue')
ax.set_xlabel('Decoding Strategy')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy by Decoding Strategy')
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 4. Prompting × Decoding Heatmap

In [ ]:
pivot = df.pivot_table(values='accuracy', index='prompting', columns='decoding', aggfunc='mean')

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.2%', cmap='YlGnBu', ax=ax, vmin=0, vmax=1)
ax.set_title('Accuracy Heatmap: Prompting × Decoding')
plt.tight_layout()
plt.show()

## 5. Dataset Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

dataset_prompting = df.pivot_table(values='accuracy', index='dataset', columns='prompting', aggfunc='mean')
dataset_prompting.plot(kind='bar', ax=ax, width=0.8)
ax.set_xlabel('Dataset')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy by Dataset and Prompting Strategy')
ax.legend(title='Prompting')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 6. Reasoning Steps vs Accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for prompting in df['prompting'].unique():
    subset = df[df['prompting'] == prompting]
    ax.scatter(subset['avg_reasoning_steps'], subset['accuracy'], label=prompting, s=100, alpha=0.7)

ax.set_xlabel('Average Reasoning Steps')
ax.set_ylabel('Accuracy')
ax.set_title('Reasoning Depth vs Accuracy')
ax.legend(title='Prompting')

plt.tight_layout()
plt.show()

## 7. Qualitative Analysis - Sample Outputs

In [ ]:
# Load detailed predictions for qualitative analysis
def load_predictions(results_dir, experiment_pattern):
    """Load prediction details for a specific experiment."""
    results_path = Path(results_dir)
    for file_path in results_path.glob(f"*{experiment_pattern}*_predictions.json"):
        with open(file_path) as f:
            return json.load(f)
    return None

# Example: compare outputs from different strategies
# predictions_cot = load_predictions(RESULTS_DIR, 'cot_greedy')
# predictions_direct = load_predictions(RESULTS_DIR, 'direct_greedy')

# if predictions_cot and predictions_direct:
#     print("Example comparison:")
#     print("\nCoT output:")
#     print(predictions_cot[0]['raw_output'][:500])
#     print("\nDirect output:")
#     print(predictions_direct[0]['raw_output'][:500])

## 8. Statistical Significance Testing

In [ ]:
from scipy import stats

# Compare best prompting strategies
prompting_strategies = df['prompting'].unique()

print("Pairwise t-tests (Prompting Strategies):")
print("-" * 50)

for i, p1 in enumerate(prompting_strategies):
    for p2 in prompting_strategies[i+1:]:
        acc1 = df[df['prompting'] == p1]['accuracy']
        acc2 = df[df['prompting'] == p2]['accuracy']
        
        if len(acc1) > 1 and len(acc2) > 1:
            t_stat, p_value = stats.ttest_ind(acc1, acc2)
            sig = "*" if p_value < 0.05 else ""
            print(f"{p1} vs {p2}: t={t_stat:.3f}, p={p_value:.4f} {sig}")

## 9. Export Results

In [ ]:
# Export summary table
summary_table = df.groupby(['prompting', 'decoding']).agg({
    'accuracy': ['mean', 'std'],
    'avg_reasoning_steps': 'mean',
    'hallucination_rate': 'mean'
}).round(4)

summary_table.to_csv('summary_table.csv')
print("Summary exported to summary_table.csv")
summary_table